1. 문서내용읽기
2. 문서쪼개기
3. 임베딩 -> 백터 데이터베이스에 저장
4. 질문이 있을때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax_with_markdown.docx')

document_list = loader.load_and_split(text_splitter=text_splitter)

C:\Users\u1ghd\AppData\Local\Temp\ipykernel_9344\1689568836.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [ ]:
split_documents

In [8]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

In [ ]:
%pip install langchain-chroma

In [ ]:
embedding.embed_documents(["테스트 문장입니다."])

In [4]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
%pip install --upgrade --quiet langchain-pinecone langchain-openai langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")

print(pinecone_api_key is not None)

True


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(document_list)

print("원본 문서 수:", len(document_list))
print("분할 후 문서 수:", len(splits))

원본 문서 수: 1
분할 후 문서 수: 320


In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=pinecone_api_key)

pc.create_index(
    name="tax-vector-index",
    dimension=4096,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

In [27]:
print(len(embedding.embed_query("차원 확인")))

4096


In [ ]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-vector-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(index_name)


database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

In [ ]:
from langchain_chroma import Chroma

database = Chroma.from_documents(documents=document_list, embedding=embedding,collection_name= 'chroma-tax', persist_directory="./chroma")

In [15]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query)

In [31]:
index.describe_index_stats()

{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 195}},
 'total_vector_count': 195,
 'vector_type': 'dense'}

In [30]:
retrieved_docs

[Document(id='0210edaf-a8d6-4320-af89-0ea210b14b36', metadata={'source': './tax.docx'}, page_content='2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]\n\n\n\n제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 한다. <개정 2014. 12. 23., 2015.

문서를 벡터로 변환하는 임베딩 = solar-embedding-1-large
질문에 답변하는 대화형 AI = solar-mini4

In [16]:
from dotenv import load_dotenv
from langchain_upstage import ChatUpstage

load_dotenv()

llm = ChatUpstage(model="solar-mini4")

In [23]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다
- [Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{context}

Question: {question}
""")

In [19]:
retriever = database.as_retriever(search_kwargs={'k': 4})
retriever.invoke(query)

[Document(id='01df1119-c797-416f-b377-7caf0e98af54', metadata={'text': '제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한

In [24]:
from langchain_classic.chains import RetrievalQA


qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

In [26]:
ai_message = qa_chain.invoke({"query": query})

In [27]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 **근로소득공제 → 종합소득과세표준 → 종합소득산출세액 → 세액공제 → 결정세액** 순서로 계산해야 합니다.  \n아래는 **2025년 귀속 연말정산 기준(일반적인 1인 가구, 기본공제만 적용)**으로 단순화한 계산 예시입니다.\n\n> ⚠️ 참고: 실제 소득세는 부양가족, 건강보험료, 신용카드 사용액, 연금저축, 기부금 등에 따라 크게 달라질 수 있습니다. 아래 계산은 **단일 근로자, 부양가족 없음, 기본공제만 적용, 다른 소득 없음**을 가정한 단순 계산입니다.\n\n---\n\n## 1. 근로소득공제\n\n연봉 5,000만원인 경우, 근로소득공제는 다음과 같습니다.\n\n| 총급여액 | 근로소득공제 |\n|---|---|\n| 5,000만원 이하 | 총급여의 70% - 150만원 |\n\n따라서,\n\n\\[\n5,000만원 \\times 70\\% - 150만원 = 3,500만원 - 150만원 = 3,350만원\n\\]\n\n**근로소득공제액: 3,350만원**\n\n---\n\n## 2. 근로소득 금액\n\n\\[\n5,000만원 - 3,350만원 = 1,650만원\n\\]\n\n**근로소득 금액: 1,650만원**\n\n---\n\n## 3. 인적공제 등 기본공제\n\n일반적인 1인 가구, 본인만 기본공제 대상이라고 가정하면,\n\n- 기본공제: 본인 150만원  \n- 추가 공제: 해당 없음\n\n따라서,  \n**종합소득공제 합계: 150만원**\n\n---\n\n## 4. 종합소득과세표준\n\n\\[\n근로소득 금액 1,650만원 - 기본공제 150만원 = 1,500만원\n\\]\n\n**종합소득과세표준: 1,500만원**\n\n---\n\n## 5. 종합소득 산출세액\n\n소득세법 제55조에 따른 세율표는 다음과 같습니다.\n\n| 과세표준 | 세율 |\n|---|---:|\n| 1,400만원 이하 

In [29]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()

In [30]:
new_question = dictionary_chain.invoke({"question": query})

In [31]:
new_question

'연봉 5천만원인 거주자의 소득세는 얼마인가요?'